In [2]:
import pandas as pd
import os
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, matthews_corrcoef

def get_metrics(y_true, y_pred):
    # Standard Confusion Matrix: [[TN, FP], [FN, TP]]
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall_pos = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    recall_neg = recall_score(y_true, y_pred, pos_label=0, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    # Pearson Binary Correlation (Matthews Correlation Coefficient)
    mcc = matthews_corrcoef(y_true, y_pred)
    
    return (tn, fp, fn, tp), precision, recall_pos, recall_neg, f1, mcc

def run_ddg_evaluation():
    # Configuration
    cases = [f"case{i}" for i in range(1, 7)]
    test_sets = [1, 2, 3]

    for case in cases:
        if not os.path.isdir(case):
            continue
            
        # Create logs directory inside the case folder
        log_dir = os.path.join(case, "logs")
        os.makedirs(log_dir, exist_ok=True)
        
        ddg_output_lines = [
            "Stability Prediction Evaluation (Embedding Model)",
            "Positive Class: Stable (ddG > 0, LLR > 0)",
            "Negative Class: Unstable (ddG <= 0, LLR <= 0)\n"
        ]

        for ts in test_sets:
            # UPDATED: New file naming convention for ESM1v
            file_name = f"scored_esm1v_Test_Set_{ts}_predictions.csv"
            file_path = os.path.join(case, file_name)
            
            if not os.path.exists(file_path):
                print(f"Skipping: {file_path} (not found)")
                continue
            
            df = pd.read_csv(file_path)
            
            # --- ddG Logic ---
            # Ground Truth: ddG > 0 is stable
            # Prediction: ESM1v_LLR > 0 is stable (Updated column name)
            y_true = (df['ddG'] > 0).astype(int)
            y_pred = (df['ESM1v_LLR'] > 0).astype(int)
            
            metrics = get_metrics(y_true, y_pred)
            (tn, fp, fn, tp), prec, rec_p, rec_n, f1, mcc = metrics
            
            # Format output (Matching ESM2 format exactly)
            ddg_output_lines.append(f"=== Test_Set_{ts} ===")
            ddg_output_lines.append(f"Total Pairs Evaluated: {len(df)}")
            ddg_output_lines.append(f"Confusion Matrix (TN, FP, FN, TP):")
            ddg_output_lines.append(f"[[{tn} {fp}]\n [{fn} {tp}]]")
            ddg_output_lines.append(f"Precision: {prec:.4f}")
            ddg_output_lines.append(f"Recall (Stable/Pos): {rec_p:.4f}")
            ddg_output_lines.append(f"Recall (Unstable/Neg): {rec_n:.4f}")
            ddg_output_lines.append(f"F1 Score: {f1:.4f}")
            ddg_output_lines.append(f"Pearson Correlation (MCC): {mcc:.4f}")
            ddg_output_lines.append("-" * 50 + "\n")

        # Write the final_stability_metrics.txt file (Same path and name as ESM2)
        output_file_path = os.path.join(log_dir, "final_stability_metrics.txt")
        with open(output_file_path, "w") as f:
            f.write("\n".join(ddg_output_lines))
        
        print(f"Generated: {output_file_path}")

if __name__ == "__main__":
    run_ddg_evaluation()

Generated: case1/logs/final_stability_metrics.txt
Generated: case2/logs/final_stability_metrics.txt
Generated: case3/logs/final_stability_metrics.txt
Generated: case4/logs/final_stability_metrics.txt
Generated: case5/logs/final_stability_metrics.txt
Generated: case6/logs/final_stability_metrics.txt
